In [1]:
print("Learning Path Recommendation System")
print("====================================")


Learning Path Recommendation System


In [2]:
import pandas as pd

data = {
    "Intern": ["Intern_1", "Intern_2", "Intern_3", "Intern_4", "Intern_5"],
    "Python": [5, 2, 4, 1, 5],
    "Machine Learning": [5, 3, 5, 1, 4],
    "Deep Learning": [4, 2, 5, 1, 5],
    "SQL": [2, 5, 1, 5, 2],
    "Data Science": [5, 3, 4, 4, 5]
}

df = pd.DataFrame(data)

df

,Intern,Python,Machine Learning,Deep Learning,SQL,Data Science
0,Intern_1,5,5,4,2,5
1,Intern_2,2,3,2,5,3
2,Intern_3,4,5,5,1,4
3,Intern_4,1,1,1,5,4
4,Intern_5,5,4,5,2,5


In [3]:
# Create the learning matrix
learning_matrix = df.set_index("Intern")

print("Intern Learning Matrix:")
learning_matrix

Intern Learning Matrix:


,Python,Machine Learning,Deep Learning,SQL,Data Science
Intern,,,,,
Intern_1,5,5,4,2,5
Intern_2,2,3,2,5,3
Intern_3,4,5,5,1,4
Intern_4,1,1,1,5,4
Intern_5,5,4,5,2,5


In [4]:
!pip install scikit-surprise

   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.3 MB ? eta -:--:--
   --------------- ------------------------ 0.5/1.3 MB 1.2 MB/s eta 0:00:01
   ------------------------------- -------- 1.0/1.3 MB 1.9 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 2.0 MB/s  0:00:00



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from surprise import Dataset, Reader, SVD

In [6]:
# Convert the learning matrix into a format for collaborative filtering

ratings_data = learning_matrix.reset_index().melt(
    id_vars=["Intern"],
    var_name="Course",
    value_name="Rating"
)

ratings_data.head(10)

,Intern,Course,Rating
0,Intern_1,Python,5
1,Intern_2,Python,2
2,Intern_3,Python,4
3,Intern_4,Python,1
4,Intern_5,Python,5
5,Intern_1,Machine Learning,5
6,Intern_2,Machine Learning,3
7,Intern_3,Machine Learning,5
8,Intern_4,Machine Learning,1
9,Intern_5,Machine Learning,4


In [8]:
reader = Reader(rating_scale=(1, 5))

dataset = Dataset.load_from_df(
    ratings_data[["Intern", "Course", "Rating"]],
    reader
)

In [9]:
trainset = dataset.build_full_trainset()

model = SVD(
    n_factors=10,
    random_state=42
)

model.fit(trainset)

In [10]:
def recommend_courses(intern, top_n=3):
    courses = learning_matrix.columns
    recommendations = []

    for course in courses:
        prediction = model.predict(intern, course)
        recommendations.append((course, prediction.est))

    recommendations.sort(key=lambda x: x[1], reverse=True)

    return recommendations[:top_n]


recommendations = recommend_courses("Intern_1")

print("Recommended Learning Path for Intern_1:")
for course, score in recommendations:
    print(f"{course}: predicted score = {score:.2f}")

Recommended Learning Path for Intern_1:
Data Science: predicted score = 4.05
Machine Learning: predicted score = 3.82
Python: predicted score = 3.77


In [11]:
recommendations = recommend_courses("Intern_2")

print("Recommended Learning Path for Intern_2:")

for course, score in recommendations:
    print(f"{course}: predicted score = {score:.2f}")

Recommended Learning Path for Intern_2:
Data Science: predicted score = 3.59
Machine Learning: predicted score = 3.34
Deep Learning: predicted score = 3.26


In [12]:
print("========================================")
print(" PERSONALIZED LEARNING PATH")
print("========================================")

intern = "Intern_2"

recommendations = recommend_courses(intern, top_n=3)

print(f"\nLearning Path for {intern}:\n")

for i, (course, score) in enumerate(recommendations, start=1):
    print(f"{i}. {course}  →  predicted score: {score:.2f}")

 PERSONALIZED LEARNING PATH

Learning Path for Intern_2:

1. Data Science  →  predicted score: 3.59
2. Machine Learning  →  predicted score: 3.34
3. Deep Learning  →  predicted score: 3.26
